# _Polynomial Regression_

# Importing Libraries
This code loads the essential tools needed for data manipulation, splitting datasets, scaling features, creating polynomial interactions, and training the regression model.


In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression  
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Loading Data
This code reads the raw training and testing housing datasets from their file paths into dataframes for analysis.


In [18]:
train = pd.read_csv('Polynomial Regression/House-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('Polynomial Regression/House-prices-advanced-regression-techniques/test.csv')

# Extracting Targets and Identifiers
This code saves the test house IDs for the final submission file and normalizes the target house prices using a log transformation.


In [19]:
test_ids = test['Id']
y = np.log1p(train['SalePrice'])


# Combining Data
This code removes unnecessary columns and joins the train and test data together so they can be cleaned identically.


In [20]:
x_train_raw = train.drop(columns=['Id', 'SalePrice'])
x_test_raw = test.drop(columns=['Id'])

combined = pd.concat([x_train_raw, x_test_raw], axis=0).reset_index(drop=True)


# Data Preprocessing and Feature Engineering

This section cleans the dataset by correcting column types, handling missing entries based on real-world meaning, and converting house quality text into ranked numbers.

* **Type Correction:** Converts `MSSubClass` from a number to text so the model treats house styles as categories rather than mathematical values.
* **Smart Missing Value Imputation:** Fills structural absences (like no pool or no garage) with `'None'` or `0`, while using medians and modes to repair genuine missing data.
* **Ordinal Quality Encoding:** Replaces text scales (like Excellent to Poor) with a sequential numeric range (`4` down to `-1`) so the model understands the structural hierarchy.
* **Final Safety Sweep:** Runs a catch-all loop at the bottom to automatically clean any remaining hidden blank cells, preventing model crashes.


In [21]:
combined['MSSubClass'] = combined['MSSubClass'].astype(str)

none_cols = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
    'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish', 
    'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'
]
for col in none_cols:
    if col in combined.columns:
        combined[col] = combined[col].fillna('None')

mode_cols = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType', 'MasVnrType', 'Functional', 'Utilities']
for col in mode_cols:
    if col in combined.columns:
        combined[col] = combined[col].fillna(combined[col].mode()[0] if not combined[col].mode().empty else 'None')

zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
for col in zero_cols:
    if col in combined.columns:
        combined[col] = combined[col].fillna(0)

combined['LotFrontage'] = combined['LotFrontage'].fillna(combined['LotFrontage'].median())

qual_mapping = {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'Po': 0, 'None': -1}
ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

for col in ordinal_cols:
    if col in combined.columns:
        combined[col] = combined[col].map(qual_mapping)

for col in combined.columns:
    if combined[col].isnull().any():
        if combined[col].dtype == 'object':
            combined[col] = combined[col].fillna('None')
        else:
            combined[col] = combined[col].fillna(0)


# One-Hot Encoding
This code converts the remaining unordered text categories into binary columns of 0s and 1s, allowing the model to use them mathematically without creating redundant columns.


In [22]:

processed_combined = pd.get_dummies(combined, drop_first=True, dtype=int)


# Re-splitting the Datasets
This code separates the unified dataset back into its original training and testing sets based on the length of the raw data.


In [23]:
x_train_final = processed_combined.iloc[:len(train)].copy()
x_test_final = processed_combined.iloc[len(train):].copy()


# Model Training and Validation Function

This function sets up the entire process for testing the model's accuracy. It splits the data, creates special feature combinations, balances the scales of the numbers, and measures performance.

* **Data Splitting:** Holds back 20% of the training data as a mini-test (validation) set to check how well the model predicts prices on houses it has never seen before.
* **Creating Interactions:** Multiplies and squares the most important numeric columns (`OverallQual`, `GrLivArea`, and `TotalBsmtSF`). This helps a basic linear model learn complex, curved patterns in the data.
* **Standardizing Values:** Puts all the different numbers on the same simple scale so large numbers (like square footage) do not confuse the model or drown out small numbers (like room counts).
* **Training and Scoring:** Trains a `LinearRegression` model on the data and prints the final validation error score (RMSE) to show exactly how close the predictions are to actual sale prices.


In [24]:
def evaluate_polynomial_regression(x_data, y_data):
    x_train, x_val, y_train, y_val = train_test_split(x_data, y_data, test_size=0.2, random_state=42)
    
    num_cols = ['OverallQual', 'GrLivArea', 'TotalBsmtSF']
    
    poly = PolynomialFeatures(degree=2, include_bias=False)
    
    x_train_poly = poly.fit_transform(x_train[num_cols])
    x_val_poly = poly.transform(x_val[num_cols])
    
    poly_cols = poly.get_feature_names_out(num_cols)
    x_train_poly_df = pd.DataFrame(x_train_poly, columns=poly_cols, index=x_train.index)
    x_val_poly_df = pd.DataFrame(x_val_poly, columns=poly_cols, index=x_val.index)
    
    x_train_final_poly = pd.concat([x_train.drop(columns=num_cols), x_train_poly_df], axis=1)
    x_val_final_poly = pd.concat([x_val.drop(columns=num_cols), x_val_poly_df], axis=1)
    
    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train_final_poly)
    x_val_scaled = scaler.transform(x_val_final_poly)
    
    model = LinearRegression()
    model.fit(x_train_scaled, y_train)
    
    y_pred = model.predict(x_val_scaled)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    print(f"Validation Root Mean Squared Error (RMSE): {rmse:.5f}")
    
    return model, scaler, poly, num_cols


# Running the Model Pipeline
This code executes the training function using the finalized training dataset and target prices, saving the trained model and data transformers for final testing.


In [25]:
trained_model, scaler, poly, num_cols = evaluate_pure_polynomial_regression(x_train_final, y)


# Submission File Verification
This code reads the saved submission file, prints its total row and column dimensions, and displays the first few predicted house prices to ensure the output formatting is correct.


In [26]:
sub_check = pd.read_csv('submission.csv')
print(sub_check.shape)
print(sub_check.head())
